In [ ]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

# === Load Data ===
train_path = "/Users/Haley/Desktop/WiDs Datathon/widsdatathon2025/TRAIN_NEW"
test_path = "/Users/Haley/Desktop/WiDs Datathon/widsdatathon2025/TEST"
print("Loading data...")
connectome_test = pd.read_csv(f"{test_path}/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")
labels_df = pd.read_excel(f"{train_path}/TRAINING_SOLUTIONS.xlsx")


Loading data...


In [11]:
# === Process Test Data ===
from connectome_gnn import *

print("Processing connectomes...")
test_graphs = create_test_graphs(connectome_test)
test_loader = DataLoader(test_graphs, batch_size=8, shuffle=False)

# === Load or Train Model ===
print("Loading model...")

model_wrapper = ModelWrapper("model.pt")

option = input("Enter 'train' to continue training or 'predict' to run inference: ").strip().lower()
if option == 'train':
    print("Loading training data...")
    connectome_train = pd.read_csv(f"{train_path}/TRAIN_FUNCTIONAL_CONNECTOME_MATRICES_new_36P_Pearson.csv")
    targets_train = pd.read_excel(f"{train_path}/TRAINING_SOLUTIONS.xlsx")

    print("Processing training connectomes...")
    train_graphs = data_to_tensor(connectome_train, targets_train)
    train_loader = DataLoader(train_graphs, batch_size=8, shuffle=True)
    print("Training model...")
    model_wrapper.train_model(train_loader)
    torch.save(model_wrapper.model.state_dict(), "model2.pt")
    print("Model saved to model2.pt")

elif option == 'predict':
    predictions = model_wrapper.predict(test_loader)

    # === Run Predictions ===
    # print("Running predictions...")
    # predicted_rows = []
    # with torch.no_grad():
    #     for data in test_loader:
    #         out = model(data)
    #         out = out.view(-1, 2)
    #         probs = torch.sigmoid(out)
    #         preds = (probs > 0.5).int()
    #         predicted_rows.extend(preds.tolist())

    submission_df = pd.DataFrame(predictions, columns=["ADHD_Outcome", "Sex_F"])
    print(submission_df)

else:
    print("Invalid option. Please run the script again and enter 'train' or 'predict'.")

# === Optional: Visualize a Sample ===
if matrices:
    print("Visualizing one sample connectome...")
    visualizer = Visualizer()
    visualizer.plot_matrix(matrices[0], title="Sample Connectome")

Processing connectomes...


NameError: name 'processor' is not defined

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.loader import DataLoader
from connectome_gnn import ConnectomeModel, ConnectomeProcessor, ConnectomeGraph, ModelWrapper, Visualizer

# === Load Data ===
train_path = "/Users/Haley/Desktop/WiDs Datathon/widsdatathon2025/TRAIN_NEW"
test_path = "/Users/Haley/Desktop/WiDs Datathon/widsdatathon2025/TEST"

print("Loading data...")
connectome_test = pd.read_csv(f"{test_path}/TEST_FUNCTIONAL_CONNECTOME_MATRICES.csv")
labels_df = pd.read_excel(f"{train_path}/TRAINING_SOLUTIONS.xlsx")

# === Process Test Data ===
processor = ConnectomeProcessor(size=200)
matrices = []

print("Processing connectomes...")
for i in range(len(connectome_test)):
    flattened = connectome_test.iloc[i, 1:].values  # skip participant_id
    try:
        matrix = processor.flatten_to_square_matrix(flattened)
        matrices.append(matrix)
    except ValueError as e:
        print(f"Skipping index {i}: {e}")

# === Create Graphs ===
test_graphs = [ConnectomeGraph(m).to_graph_data() for m in matrices]
test_loader = DataLoader(test_graphs, batch_size=8, shuffle=False)

# # === Load Trained Model ===
print("Loading model...")
model = ConnectomeModel()
model.load_state_dict(torch.load("model.pt", map_location=torch.device('cpu')))
model.eval()
# === Run Predictions ===
print("Running predictions...")
predicted_rows = []
with torch.no_grad():
    for data in test_loader:
        out = model(data)
        out = out.view(-1, 2)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).int()
        predicted_rows.extend(preds.tolist())

submission_df = pd.DataFrame(predicted_rows, columns=["ADHD_Outcome", "Sex_F"])
print(submission_df)

# submission_df.to_csv("submission.csv", index=False)
# print("Saved predictions to submission.csv")

# === Optional: Visualize a Sample ===
print("Visualizing one sample connectome...")
visualizer = Visualizer()
visualizer.plot_matrix(matrices[0], title="Sample Connectome")
